# 📊 Notebook 3 — Data Analyst
## Forestry Image Classification | ISB46703

**Role:** Data Analyst  
**Tasks:**
- Visualize training loss & accuracy for all models
- Display confusion matrices
- Evaluate models on test dataset
- Draw final conclusion on the best model


## 📦 1. Imports & Load Results

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import (
    confusion_matrix, classification_report,
    average_precision_score, roc_curve, auc
)

RESULTS_DIR = '../results'
BASE_DIR    = '../dataset'
CLASSES     = ['healthy_forest', 'deforested', 'forest_fire', 'flooded_forest', 'diseased_forest']
CLASS_LABELS = [c.replace('_', ' ').title() for c in CLASSES]
MODEL_NAMES = ['ResNet50', 'DenseNet121', 'MobileNetV3']
IMG_SIZE    = (224, 224)
BATCH_SIZE  = 32

# Load saved evaluation results
with open(f'{RESULTS_DIR}/eval_results.json') as f:
    eval_results = json.load(f)

# Load training histories
histories = {}
for name in MODEL_NAMES:
    path = f'{RESULTS_DIR}/{name.lower()}_history.json'
    with open(path) as f:
        data = json.load(f)
        histories[name] = data['history']

print('✅ Results loaded')
print(f'   Models  : {MODEL_NAMES}')
print(f'   Classes : {CLASSES}')

## 🔄 2. Load Test Generator & Models

In [ ]:
test_datagen = ImageDataGenerator(rescale=1./255)
test_gen = test_datagen.flow_from_directory(
    os.path.join(BASE_DIR, 'test'),
    target_size = IMG_SIZE,
    batch_size  = BATCH_SIZE,
    class_mode  = 'categorical',
    classes     = CLASSES,
    shuffle     = False
)

# Load saved models
models = {}
for name in MODEL_NAMES:
    model_path = f'{RESULTS_DIR}/{name}_best.keras'
    if os.path.exists(model_path):
        models[name] = tf.keras.models.load_model(model_path)
        print(f'✅ Loaded: {name}')
    else:
        print(f'⚠️  Not found: {model_path} — run notebook 02 first')

## 📈 3. Training Curves — Loss & Accuracy

In [ ]:
palette = {
    'ResNet50'    : ('#1ABC9C', '#16A085'),
    'DenseNet121' : ('#3498DB', '#2980B9'),
    'MobileNetV3' : ('#E67E22', '#D35400')
}

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('🌲 Forestry CNN — Training Curves (50 Epochs)', fontsize=16, fontweight='bold', y=1.01)

for col, name in enumerate(MODEL_NAMES):
    hist   = histories[name]
    ep     = range(1, len(hist['accuracy']) + 1)
    c1, c2 = palette[name]
    
    # ── ACCURACY ────────────────────────────────────────────────
    ax_acc = axes[0][col]
    ax_acc.plot(ep, hist['accuracy'],     color=c1, lw=2, label='Train')
    ax_acc.plot(ep, hist['val_accuracy'], color=c2, lw=2, ls='--', label='Validation')
    ax_acc.axvline(25, color='gray', ls=':', alpha=0.6, label='Fine-tune')
    ax_acc.fill_between(ep, hist['accuracy'], hist['val_accuracy'], alpha=0.08, color=c1)
    ax_acc.set_title(f'{name}\nAccuracy', fontweight='bold')
    ax_acc.set_xlabel('Epoch')
    ax_acc.set_ylabel('Accuracy')
    ax_acc.legend(fontsize=9)
    ax_acc.grid(alpha=0.3)
    ax_acc.set_ylim(0, 1)
    best_acc = max(hist['val_accuracy'])
    ax_acc.annotate(f'Best: {best_acc:.3f}',
                    xy=(hist['val_accuracy'].index(best_acc) + 1, best_acc),
                    xytext=(5, -15), textcoords='offset points',
                    fontsize=9, color=c2,
                    arrowprops=dict(arrowstyle='->', color=c2, lw=1))
    
    # ── LOSS ────────────────────────────────────────────────────
    ax_loss = axes[1][col]
    ax_loss.plot(ep, hist['loss'],     color=c1, lw=2, label='Train')
    ax_loss.plot(ep, hist['val_loss'], color=c2, lw=2, ls='--', label='Validation')
    ax_loss.axvline(25, color='gray', ls=':', alpha=0.6, label='Fine-tune')
    ax_loss.set_title(f'{name}\nLoss', fontweight='bold')
    ax_loss.set_xlabel('Epoch')
    ax_loss.set_ylabel('Loss')
    ax_loss.legend(fontsize=9)
    ax_loss.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/training_curves_detailed.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: results/training_curves_detailed.png')

## 🔲 4. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 7))
fig.suptitle('🔲 Confusion Matrices — Test Set', fontsize=16, fontweight='bold')

cmaps = ['Greens', 'Blues', 'Oranges']

for idx, (name, model) in enumerate(models.items()):
    test_gen.reset()
    y_pred_probs = model.predict(test_gen, verbose=0)
    y_pred       = np.argmax(y_pred_probs, axis=1)
    y_true       = test_gen.classes
    
    cm  = confusion_matrix(y_true, y_pred)
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
    
    ax = axes[idx]
    sns.heatmap(
        cm_pct, annot=True, fmt='.1f', cmap=cmaps[idx],
        xticklabels=CLASS_LABELS, yticklabels=CLASS_LABELS,
        ax=ax, linewidths=0.5,
        cbar_kws={'label': '% of True Class'}
    )
    acc = np.trace(cm) / np.sum(cm)
    ax.set_title(f'{name}\nTest Accuracy: {acc:.2%}', fontweight='bold')
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')
    ax.tick_params(axis='x', rotation=30)
    ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: results/confusion_matrices.png')

## 🎯 5. Per-Class Performance (mAP per Class)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

x    = np.arange(len(CLASSES))
w    = 0.25
bars_colors = ['#1ABC9C', '#3498DB', '#E67E22']

for i, name in enumerate(MODEL_NAMES):
    per_ap = eval_results[name]['per_class_ap']
    bars = ax.bar(x + i*w, per_ap, w, label=name,
                  color=bars_colors[i], edgecolor='white', linewidth=0.5)
    for bar, val in zip(bars, per_ap):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                f'{val:.2f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x + w)
ax.set_xticklabels(CLASS_LABELS, rotation=20, ha='right')
ax.set_ylabel('Average Precision (AP)')
ax.set_title('Per-Class Average Precision — All Models', fontweight='bold', fontsize=14)
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 1.1)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/per_class_ap.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: results/per_class_ap.png')

## 🏆 6. Model Comparison Dashboard

In [ ]:
accs   = [eval_results[n]['test_accuracy']  for n in MODEL_NAMES]
maps   = [eval_results[n]['mAP']            for n in MODEL_NAMES]
times  = [eval_results[n]['training_time']  for n in MODEL_NAMES]
params = [eval_results[n]['params_M']       for n in MODEL_NAMES]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('🏆 Model Comparison Dashboard', fontsize=16, fontweight='bold')
colors = ['#1ABC9C', '#3498DB', '#E67E22']

def make_bar(ax, values, title, ylabel, highlight_max=True):
    max_val = max(values)
    bar_colors = [('#E74C3C' if v == max_val else c) if highlight_max
                  else ('#E74C3C' if v == min(values) else c)
                  for v, c in zip(values, colors)]
    bars = ax.bar(MODEL_NAMES, values, color=bar_colors, edgecolor='white', linewidth=1)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() * 0.98,
                f'{val:.3f}' if val < 10 else f'{val:.1f}',
                ha='center', va='top', fontweight='bold', color='white', fontsize=11)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel(ylabel)
    ax.grid(axis='y', alpha=0.3)

make_bar(axes[0][0], accs,   'Test Accuracy  (↑ Higher = Better)', 'Accuracy')
make_bar(axes[0][1], maps,   'Mean Average Precision  (↑ Higher = Better)', 'mAP')
make_bar(axes[1][0], times,  'Training Time  (↓ Lower = Better)', 'Minutes', highlight_max=False)
make_bar(axes[1][1], params, 'Model Parameters  (↓ Lower = Lighter)', 'Millions (M)', highlight_max=False)

# Add legend
legend_patches = [
    mpatches.Patch(color='#E74C3C', label='🥇 Best in category'),
    mpatches.Patch(color='#1ABC9C', label='ResNet50'),
    mpatches.Patch(color='#3498DB', label='DenseNet121'),
    mpatches.Patch(color='#E67E22', label='MobileNetV3'),
]
fig.legend(handles=legend_patches, loc='lower center', ncol=4, fontsize=10, frameon=True, bbox_to_anchor=(0.5, -0.03))

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/model_comparison_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: results/model_comparison_dashboard.png')

## 📋 7. Classification Report

In [ ]:
for name, model in models.items():
    test_gen.reset()
    y_pred = np.argmax(model.predict(test_gen, verbose=0), axis=1)
    y_true = test_gen.classes
    
    print(f'\n{"="*60}')
    print(f'📋 Classification Report — {name}')
    print('='*60)
    print(classification_report(y_true, y_pred, target_names=CLASS_LABELS, digits=4))

## 🏁 8. Final Conclusion

In [ ]:
# ── SCORING: rank models across 4 criteria ──────────────────────
# Higher accuracy → better | Higher mAP → better
# Lower time → better      | Fewer params → better

def rank_score(values, ascending=True):
    """Return normalized score (0–1) where 1 = best."""
    arr = np.array(values, dtype=float)
    mn, mx = arr.min(), arr.max()
    if mx == mn:
        return np.ones(len(values))
    norm = (arr - mn) / (mx - mn)
    return norm if not ascending else 1 - norm   # ascending=True means lower is better

acc_scores   = rank_score(accs,   ascending=False)  # higher acc = better → ascending=False
map_scores   = rank_score(maps,   ascending=False)
time_scores  = rank_score(times,  ascending=True)   # lower time = better
param_scores = rank_score(params, ascending=True)

# Weighted composite score (weights reflect task priority)
w_acc, w_map, w_time, w_param = 0.35, 0.35, 0.20, 0.10
composite = (w_acc*acc_scores + w_map*map_scores + w_time*time_scores + w_param*param_scores)

best_idx   = np.argmax(composite)
best_model = MODEL_NAMES[best_idx]

print('=' * 65)
print('🏁  FINAL CONCLUSION — Forestry Image Classification')
print('=' * 65)
print()
print(f'  {'Model':<14} {'Accuracy':>10} {'mAP':>10} {'Time(min)':>10} {'Params(M)':>10} {'Score':>8}')
print(f'  {'-'*62}')
for i, name in enumerate(MODEL_NAMES):
    marker = '  ◀ BEST' if i == best_idx else ''
    print(f"  {name:<14} {accs[i]:>10.4f} {maps[i]:>10.4f} {times[i]:>10.1f} {params[i]:>10.1f} {composite[i]:>8.4f}{marker}")

print()
print(f'🥇 Best Model: {best_model}')
print()
print('── Rationale ─────────────────────────────────────────────')

rationale = {
    'ResNet50': (
        'ResNet50 achieved the highest overall classification accuracy and mAP '
        'among all three models, demonstrating strong representational capacity '
        'through its residual connections. While it has the most parameters (25.6M) '
        'and longer training time, its performance advantage makes it the most '
        'suitable model for this forestry classification task where precision is critical.'
    ),
    'DenseNet121': (
        'DenseNet121 delivered the best balance between accuracy and efficiency. '
        'Its dense connectivity pattern encourages feature reuse, yielding competitive '
        'mAP scores with fewer parameters than ResNet50. It is the recommended choice '
        'when both accuracy and deployment efficiency are considered together.'
    ),
    'MobileNetV3': (
        'MobileNetV3 stands out for its efficiency — fewest parameters (5.4M) and '
        'fastest training time — making it ideal for deployment on edge devices or '
        'embedded forestry monitoring systems. While its raw accuracy is slightly lower, '
        'its speed and size advantage make it best suited for real-time applications.'
    )
}

print(rationale[best_model])
print()
print('── Trade-off Summary ─────────────────────────────────────')
print('  • For maximum accuracy & mAP     → ResNet50')
print('  • For accuracy-efficiency balance → DenseNet121')
print('  • For speed & lightweight deploy  → MobileNetV3')
print('=' * 65)

## 📊 9. Radar Chart — Multi-criteria Comparison

In [ ]:
from matplotlib.patches import FancyArrowPatch

categories   = ['Accuracy', 'mAP', 'Speed\n(inverse time)', 'Efficiency\n(inverse params)']
N            = len(categories)
angles       = [n / float(N) * 2 * np.pi for n in range(N)]
angles      += angles[:1]  # close the polygon

model_scores = {
    MODEL_NAMES[0]: [acc_scores[0],  map_scores[0],  time_scores[0],  param_scores[0]],
    MODEL_NAMES[1]: [acc_scores[1],  map_scores[1],  time_scores[1],  param_scores[1]],
    MODEL_NAMES[2]: [acc_scores[2],  map_scores[2],  time_scores[2],  param_scores[2]],
}

colors_r = ['#1ABC9C', '#3498DB', '#E67E22']

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
fig.suptitle('Multi-Criteria Radar Chart', fontsize=14, fontweight='bold')

for (name, scores), color in zip(model_scores.items(), colors_r):
    vals  = scores + scores[:1]
    ax.plot(angles, vals, 'o-', linewidth=2, label=name, color=color)
    ax.fill(angles, vals, alpha=0.1, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=11)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['0.25', '0.50', '0.75', '1.00'], fontsize=8)
ax.grid(True, alpha=0.3)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.15))

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/radar_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: results/radar_comparison.png')

---
## ✅ Analysis Complete!

### 📁 All outputs saved to `results/`:
| File | Content |
|------|---------|
| `dataset_distribution.png` | Class distribution & split sizes |
| `sample_images.png` | Sample images per class |
| `training_curves_detailed.png` | Loss & accuracy per model |
| `confusion_matrices.png` | Confusion matrices on test set |
| `per_class_ap.png` | Per-class AP scores |
| `model_comparison_dashboard.png` | Side-by-side metric comparison |
| `radar_comparison.png` | Multi-criteria radar chart |
| `eval_results.json` | Numeric results (accuracy, mAP, time) |

**Presentation ready!** 🎉